# Computational Lab Notebook

**Project:** [short project name]
**Author:** Your Name
**Started:** YYYY-MM-DD

---

**How to use this template.** A computational lab notebook plays the same
role as a wet-lab notebook: a dated, honest record of what you did, why you
did it, what happened, and what you concluded — including the dead ends.

Suggested workflow:

1. Copy this file and rename it, e.g. `2026-10-06_rnaseq-qc.ipynb`
   (one notebook per experiment or analysis session).
2. Fill in each section **as you work**, not after the fact.
3. Before saving a final version, use *Restart Kernel and Run All Cells* so
   the saved outputs really come from the code shown, in order.
4. Commit the notebook to version control.

Everything below the **Worked example** line is a filled-in entry using the
course stickleback dataset — replace it with your own.

## Entry: YYYY-MM-DD — [one-line title of this analysis]

### 1. Objective

*What question is this entry trying to answer? One or two sentences. If you
can't state the objective, you aren't ready to write code.*

### 2. Background and rationale

*Why are you doing this now? Link to prior entries, papers, or lab
discussions. What did you already know before starting?*

### 3. Data provenance

*Exactly where the data came from — future-you will not remember.*

| Item | Value |
|------|-------|
| File(s) | `path/to/data.tsv` |
| Source | e.g. sequencing core, collaborator, NCBI accession |
| Downloaded / received | YYYY-MM-DD |
| Version / md5 checksum | `md5sum path/to/data.tsv` |

### 4. Methods and parameters

*Software, versions, and every non-default parameter. The code cells below
are part of the record, but summarize key choices in prose.*

### 5. Analysis

*Interleave narrative and code. Say what each cell is about to do and, after
it runs, what you see in the output.*

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### 6. Results and interpretation

*What did you find? Refer to specific numbers and figures. Separate
observation ("the interaction term is significant, p = 3e-7") from
interpretation ("the microbiota effect depends on host genotype").*

### 7. Problems and dead ends

*Things that failed, surprised you, or were abandoned — and why. This is the
section that makes a notebook honest and useful.*

### 8. Next steps

*Concrete, actionable items for the next session.*

### 9. Reproducibility record

*Run the version cell at the end of every session (see the bottom of the
worked example).*

---

## Worked example entry: 2026-09-29 — Genotype and microbiota effects on Gene369

### 1. Objective

Test whether gut expression of Gene369 depends on host genotype, microbiota
treatment, and their interaction, in the course RNA-seq dataset of 80
stickleback.

### 2. Background and rationale

Gene369 showed high between-group variance in an initial screen of the
600-gene count table. The experiment crossed 2 host genotypes (A, B) with 2
microbiota treatments (conventional, mono-associated), n = 20 fish per
group, so a two-factor linear model is the natural first analysis.

### 3. Data provenance

| Item | Value |
|------|-------|
| File | `Data_Folder/Stickle_RNAseq.tsv` |
| Source | Course website (Cresko Lab gut RNA-seq counts) |
| Downloaded | 2026-09-29 |
| Shape | 80 fish x 604 columns (3 factors + 600 genes) |

### 4. Methods and parameters

Counts were normalized to log2 counts-per-million (CPM) with a pseudocount
of 1, using each fish's total count as its library size. Effects were tested
with an ordinary least-squares linear model,
`gene369 ~ Genotype * Microbiota`, using `statsmodels`
(install once with `pip install statsmodels`).

### 5. Analysis

Read the counts and normalize the focal gene:

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Look for the data locally first; fall back to the course website.
local_paths = [
    Path("Data_Folder/Stickle_RNAseq.tsv"),
    Path("../Data_Folder/Stickle_RNAseq.tsv"),
]
found = [p for p in local_paths if p.exists()]
if found:
    data_file = found[0]
else:
    data_file = "https://wcresko.github.io/BioE_Comp_Tools_Fa2026/Data_Folder/Stickle_RNAseq.tsv"

stickle = pd.read_csv(data_file, sep="\t")
gene_cols = [c for c in stickle.columns if c.startswith("Gene")]
stickle["lib_size"] = stickle[gene_cols].sum(axis=1)

focal = stickle[["Individual", "Genotype", "Microbiota"]].copy()
focal["gene369"] = np.log2(stickle["Gene369"] / stickle["lib_size"] * 1e6 + 1)

focal.value_counts(["Genotype", "Microbiota"])

Visualize expression by group before modeling:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(
    data=focal, x="Microbiota", y="gene369", hue="Genotype",
    showfliers=False, boxprops={"alpha": 0.6}, ax=ax,
)
sns.stripplot(
    data=focal, x="Microbiota", y="gene369", hue="Genotype",
    dodge=True, size=4, alpha=0.6, legend=False, ax=ax,
)
ax.set_ylabel("Gene369 expression (log2 CPM)")
ax.set_title("Gene369 by genotype and microbiota treatment")
fig.tight_layout()

Fit the two-factor model with interaction:

In [ ]:
fit = smf.ols("gene369 ~ Genotype * Microbiota", data=focal).fit()
print(fit.summary())
print(sm.stats.anova_lm(fit))

Check the residuals:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(fit.fittedvalues, fit.resid, alpha=0.7)
ax.axhline(0, color="gray", linestyle="--")
ax.set_xlabel("Fitted values")
ax.set_ylabel("Residuals")
ax.set_title("Residuals vs. fitted")
fig.tight_layout()

### 6. Results and interpretation

Both main effects and the interaction are significant. Genotype B fish
express Gene369 about 4.5 log2 CPM units lower than genotype A (p < 1e-30),
mono-association lowers expression by about 2.8 units in genotype A
(p < 1e-20), and the positive interaction term (+1.5, p = 3e-7) means the
microbiota effect is weaker in genotype B. The model explains ~93% of the
variance. Interpretation: the transcriptional response of Gene369 to
microbiota depends on host genotype — a candidate for
genotype-by-environment interaction.

### 7. Problems and dead ends

Raw counts were initially modeled without normalization; library sizes vary
~4.6-fold across fish, which inflated the residual variance. Switched to
log2 CPM.

### 8. Next steps

- Repeat the model across all 600 genes; adjust p-values (FDR).
- Check whether Gene369's co-expression partners (e.g. Gene208) show the
  same interaction pattern.

### 9. Reproducibility record

In [ ]:
import sys
import matplotlib
import statsmodels

print("Python:", sys.version)
for name, mod in [("numpy", np), ("pandas", pd), ("seaborn", sns),
                  ("matplotlib", matplotlib), ("statsmodels", statsmodels)]:
    print(name, mod.__version__)